In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.power import TTestIndPower

In [3]:
df = pd.read_csv('Base_de_Datos_dynamic_pricing.csv')

In [4]:
grupo1 = df[df['Vehicle_Type'] == 'Economy']['Historical_Cost_of_Ride'].dropna()
grupo2 = df[df['Vehicle_Type'] == 'Premium']['Historical_Cost_of_Ride'].dropna()

n1, n2 = len(grupo1), len(grupo2)
media1, media2 = grupo1.mean(), grupo2.mean()
sd1, sd2 = grupo1.std(ddof=1), grupo2.std(ddof=1)

print(f"Economy : n={n1}, media={media1:.2f}, sd={sd1:.2f}")
print(f"Premium : n={n2}, media={media2:.2f}, sd={sd2:.2f}")

Economy : n=478, media=346.57, sd=186.99
Premium : n=522, media=396.25, sd=184.32


In [5]:
t_stat, p_valor = stats.ttest_ind(grupo1, grupo2, equal_var=False)
print(f"\nt = {t_stat:.4f}, p-valor = {p_valor:.4g}")


t = -4.2249, p-valor = 2.611e-05


In [6]:
gl = n1 + n2 - 2
var_pooled = ((n1-1)*sd1**2 + (n2-1)*sd2**2) / gl
sd_pooled = np.sqrt(var_pooled)
cohens_d = abs(media1 - media2) / sd_pooled

print(f"Cohen's d = {cohens_d:.4f}")

Cohen's d = 0.2676


In [7]:
alpha = 0.05
analisis = TTestIndPower()
poder = analisis.solve_power(
    effect_size=cohens_d,
    nobs1=n1,
    alpha=alpha,
    ratio=n2/n1,
    alternative='two-sided'
)
print(f"Poder estadístico = {poder:.4f} ({poder*100:.1f}%)")
print(f"Beta (riesgo Error Tipo II) = {1-poder:.4f}")

Poder estadístico = 0.9882 (98.8%)
Beta (riesgo Error Tipo II) = 0.0118


In [8]:
n_necesario = analisis.solve_power(
    effect_size=cohens_d, alpha=alpha, power=0.80,
    ratio=n2/n1, alternative='two-sided'
)
print(f"n por grupo necesario para poder 0.80: {n_necesario:.1f}")

n por grupo necesario para poder 0.80: 210.8


## Interpretación del análisis

En este análisis se comparó el `Historical_Cost_of_Ride` entre los vehículos de tipo **Economy** y **Premium**. El grupo Economy estuvo conformado por 478 observaciones, con un costo histórico promedio de **346.57** y una desviación estándar de **186.99**. El grupo Premium contó con 522 observaciones, presentó un promedio de **396.25** y una desviación estándar de **184.32**. Descriptivamente, el costo promedio de los viajes Premium fue aproximadamente **49.68 unidades monetarias mayor** que el de los viajes Economy.

Para evaluar si esta diferencia podía atribuirse al azar del muestreo se aplicó una prueba t de Welch, adecuada cuando se comparan las medias de dos grupos independientes sin asumir que sus varianzas poblacionales son exactamente iguales. Las hipótesis planteadas fueron:

- **H₀:** el costo histórico promedio es igual para los vehículos Economy y Premium (`μEconomy = μPremium`).
- **H₁:** los costos históricos promedio son diferentes (`μEconomy ≠ μPremium`).

El estadístico obtenido fue **t = -4.2249** y el p-valor fue **2.611 × 10⁻⁵**, valor considerablemente menor que el nivel de significancia de 0.05. Por lo tanto, se rechaza la hipótesis nula y existe evidencia estadística suficiente para afirmar que el costo histórico promedio difiere entre ambos tipos de vehículo. Según las medias observadas, los viajes Premium presentan un costo histórico promedio superior al de los viajes Economy.

Además de la significancia estadística, se calculó el tamaño del efecto mediante la d de Cohen, cuyo valor fue **0.2676**. Este resultado representa una diferencia de magnitud pequeña según las referencias habituales, aunque su importancia práctica debe valorarse junto con el contexto económico del servicio. La diferencia es estadísticamente significativa en parte porque el análisis cuenta con una muestra amplia, pero el tamaño del efecto indica que existe una considerable superposición entre los costos individuales de ambos grupos.

El poder estadístico estimado fue de **98.8 %**, mientras que el riesgo de cometer un error de tipo II fue aproximadamente **1.18 %**. Esto significa que, bajo las condiciones y el tamaño del efecto estimado, el estudio tenía una alta probabilidad de detectar una diferencia real entre los grupos. Para alcanzar un poder aproximado del 80 % habría sido suficiente contar con cerca de **211 observaciones por grupo**, mientras que la base utilizada supera ese tamaño en ambos tipos de vehículo.
